# 🧩 Mini-Lab: Checkpoint v0

**Module: Ship a Working AI Feature Today** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Produce** a v0 checkpoint artifact that captures your pipeline's current state — what works, what breaks, and how accurate it is
2. **Compile** a written list of open questions that guide your next iteration
3. **Combine** pipeline-stage awareness, traceback reading, and baseline comparison into a single checkpoint document

## Target Concepts

| Concept | Gloss |
|---------|-------|
| AI Pipeline Stages | A practical AI feature pipeline with six boxes: ingest, prepare, call, validate, log, evaluate. The model call is just one box, not the whole system. |
| Traceback Reading | Read the last line first to see the error type and message, then scan upward to find the line that names your file, that is usually where you should start fixing. |
| Baseline Comparison | Judge output against a simple reference baseline (a cheap, clear alternative), not against "looks good". If you cannot beat the baseline, you have not earned complexity. |


## What Is a v0 Checkpoint?

A **v0 checkpoint** is the first snapshot of your AI pipeline that answers three questions:

| Question | How You Answer It |
|----------|-------------------|
| **What works?** | List which pipeline stages run successfully |
| **What breaks?** | Record errors you hit, mapped to their stage |
| **How good is it?** | Show your baseline spot-check score |

Plus one forward-looking piece:

- **Open questions** — things you don't know yet that would affect the next version

This is the artifact you'd share with a teammate or manager to say: *"Here's where we are after the first pass."*

In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## Step 1: Record Pipeline Stage Status

Recall the six stages from `mini-map-pipeline-stages`. For your v0 checkpoint, you walk through each stage and record whether it ran successfully, failed, or wasn't attempted yet.

Let's simulate the status of a ticket-triage pipeline after a first pass (like the one you built in `lab-triage-csv-rows`).

In [2]:
# Record the status of each pipeline stage after your first end-to-end run
stage_status = [
    {"stage": 1, "name": "Ingest",       "status": "✅ Pass", "notes": "CSV loads correctly, 6 tickets read"},
    {"stage": 2, "name": "Prepare",      "status": "✅ Pass", "notes": "Empty messages skipped, 6/6 passed"},
    {"stage": 3, "name": "Prompt",       "status": "✅ Pass", "notes": "System + user messages built for each ticket"},
    {"stage": 4, "name": "Call Model",   "status": "✅ Pass", "notes": "gpt-4.1-nano, temperature=0"},
    {"stage": 5, "name": "Parse Output", "status": "⚠️ Fragile", "notes": "Works now, but no fallback if model returns non-JSON"},
    {"stage": 6, "name": "Deliver",      "status": "✅ Pass", "notes": "Results collected into a list of dicts"},
]

header = "| Stage | Name | Status | Notes |\n|-------|------|--------|-------|\n"
rows = ""
for s in stage_status:
    rows += f"| {s['stage']} | {s['name']} | {s['status']} | {s['notes']} |\n"

md("### Pipeline Stage Status\n\n" + header + rows)

### Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------|-------|
| 1 | Ingest | ✅ Pass | CSV loads correctly, 6 tickets read |
| 2 | Prepare | ✅ Pass | Empty messages skipped, 6/6 passed |
| 3 | Prompt | ✅ Pass | System + user messages built for each ticket |
| 4 | Call Model | ✅ Pass | gpt-4.1-nano, temperature=0 |
| 5 | Parse Output | ⚠️ Fragile | Works now, but no fallback if model returns non-JSON |
| 6 | Deliver | ✅ Pass | Results collected into a list of dicts |


## Step 2: Record Errors Encountered (with Stage Labels)

From `mini-read-tracebacks`, you learned to read errors bottom-up and identify the stage. Your v0 checkpoint should list every error you hit during development, mapped to its stage. This creates a debugging log for your future self.

In [3]:
# Log of errors encountered during development, mapped to pipeline stages
errors_encountered = [
    {
        "stage": 1,
        "stage_name": "Ingest",
        "error_type": "FileNotFoundError",
        "description": "Typo in CSV path ('data/tickers.csv' instead of 'data/tickets.csv')",
        "resolved": True,
    },
    {
        "stage": 3,
        "stage_name": "Prompt",
        "error_type": "KeyError",
        "description": "Referenced ticket['subject'] instead of ticket['message']",
        "resolved": True,
    },
    {
        "stage": 5,
        "stage_name": "Parse Output",
        "error_type": "JSONDecodeError",
        "description": "Model occasionally returns 'Sure! Here is...' prefix before JSON",
        "resolved": False,
    },
]

header = "| Stage | Error Type | Description | Resolved? |\n|-------|------------|-------------|-----------|\n"
rows = ""
for e in errors_encountered:
    icon = "✅" if e["resolved"] else "❌"
    rows += f"| {e['stage']} — {e['stage_name']} | `{e['error_type']}` | {e['description']} | {icon} |\n"

resolved_count = sum(1 for e in errors_encountered if e["resolved"])
total_count = len(errors_encountered)

md(
    f"### Errors Encountered\n\n{header}{rows}\n"
    f"**{resolved_count}/{total_count} resolved** — {total_count - resolved_count} still open"
)

### Errors Encountered

| Stage | Error Type | Description | Resolved? |
|-------|------------|-------------|-----------|
| 1 — Ingest | `FileNotFoundError` | Typo in CSV path ('data/tickers.csv' instead of 'data/tickets.csv') | ✅ |
| 3 — Prompt | `KeyError` | Referenced ticket['subject'] instead of ticket['message'] | ✅ |
| 5 — Parse Output | `JSONDecodeError` | Model occasionally returns 'Sure! Here is...' prefix before JSON | ❌ |

**2/3 resolved** — 1 still open

## Step 3: Record Baseline Accuracy

From `mini-baseline-spot-check`, you know that "looks good" isn't enough. Your v0 checkpoint records the actual score against your labelled baseline.

In [4]:
# Baseline results from your spot-check (reusing the data from mini-baseline-spot-check)
baseline_results = [
    {"ticket_id": "101", "expected": "Billing",   "predicted": "Billing",   "match": True},
    {"ticket_id": "102", "expected": "Technical", "predicted": "Technical", "match": True},
    {"ticket_id": "103", "expected": "Account",   "predicted": "General",   "match": False},
    {"ticket_id": "104", "expected": "General",   "predicted": "General",   "match": True},
    {"ticket_id": "105", "expected": "Billing",   "predicted": "Billing",   "match": True},
]

correct = sum(1 for r in baseline_results if r["match"])
total = len(baseline_results)
accuracy = correct / total

header = "| Ticket | Expected | Predicted | Match |\n|--------|----------|-----------|-------|\n"
rows = ""
for r in baseline_results:
    icon = "✅" if r["match"] else "❌"
    rows += f"| {r['ticket_id']} | {r['expected']} | {r['predicted']} | {icon} |\n"

md(
    f"### Baseline Spot-Check\n\n{header}{rows}\n"
    f"**Accuracy: {correct}/{total} ({accuracy:.0%})**"
)

### Baseline Spot-Check

| Ticket | Expected | Predicted | Match |
|--------|----------|-----------|-------|
| 101 | Billing | Billing | ✅ |
| 102 | Technical | Technical | ✅ |
| 103 | Account | General | ❌ |
| 104 | General | General | ✅ |
| 105 | Billing | Billing | ✅ |

**Accuracy: 4/5 (80%)**

## Step 4: Write Open Questions

The most underrated part of a checkpoint is the **open questions list**. These are things you don't know yet — unknowns that would change your approach if you had answers. Writing them down prevents you from forgetting and gives your team a clear picture of what remains uncertain.

In [5]:
# Open questions — things you'd want to answer before v1
open_questions = [
    "The model confused 'Account' with 'General' on ticket 103. Would adding a few-shot example for Account changes fix this?",
    "Parse Output has no fallback — what happens if the model wraps JSON in markdown fences (```json ... ```)?",
    "We tested on 5 tickets. Is 5 enough to trust the accuracy, or do we need 20+?",
    "Should we add a 'confidence' threshold and route low-confidence tickets to a human?",
    "The current pipeline processes tickets one at a time. Will this be too slow for 1,000 tickets?",
]

output = "### Open Questions\n\n"
for i, q in enumerate(open_questions, 1):
    output += f"{i}. {q}\n"

md(output)

### Open Questions

1. The model confused 'Account' with 'General' on ticket 103. Would adding a few-shot example for Account changes fix this?
2. Parse Output has no fallback — what happens if the model wraps JSON in markdown fences (```json ... ```)?
3. We tested on 5 tickets. Is 5 enough to trust the accuracy, or do we need 20+?
4. Should we add a 'confidence' threshold and route low-confidence tickets to a human?
5. The current pipeline processes tickets one at a time. Will this be too slow for 1,000 tickets?


## Step 5: Assemble the v0 Checkpoint

Now we combine all four pieces into a single checkpoint artifact — one document you could save, print, or paste into a team thread.

In [6]:
from datetime import date

# --- Build the full checkpoint document ---
today = date.today().isoformat()

# Section 1: Header
checkpoint_text = f"""# v0 Checkpoint: Ticket Triage Pipeline

**Date:** {today}  
**Author:** AI Engineer  
**Pipeline:** Customer support ticket classification  
**Model:** gpt-4.1-nano (temperature=0)

---

## 1. Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------|-------|
"""
for s in stage_status:
    checkpoint_text += f"| {s['stage']} | {s['name']} | {s['status']} | {s['notes']} |\n"

# Section 2: Errors
checkpoint_text += f"\n## 2. Errors Encountered ({resolved_count}/{total_count} resolved)\n\n"
checkpoint_text += "| Stage | Error Type | Description | Resolved? |\n|-------|------------|-------------|-----------|\n"
for e in errors_encountered:
    icon = "✅" if e["resolved"] else "❌"
    checkpoint_text += f"| {e['stage']} — {e['stage_name']} | `{e['error_type']}` | {e['description']} | {icon} |\n"

# Section 3: Baseline
checkpoint_text += f"\n## 3. Baseline Accuracy: {correct}/{total} ({accuracy:.0%})\n\n"
checkpoint_text += "| Ticket | Expected | Predicted | Match |\n|--------|----------|-----------|-------|\n"
for r in baseline_results:
    icon = "✅" if r["match"] else "❌"
    checkpoint_text += f"| {r['ticket_id']} | {r['expected']} | {r['predicted']} | {icon} |\n"

# Section 4: Open Questions
checkpoint_text += "\n## 4. Open Questions\n\n"
for i, q in enumerate(open_questions, 1):
    checkpoint_text += f"{i}. {q}\n"

# Display the full checkpoint
md(checkpoint_text)

# v0 Checkpoint: Ticket Triage Pipeline

**Date:** 2026-09-16  
**Author:** AI Engineer  
**Pipeline:** Customer support ticket classification  
**Model:** gpt-4.1-nano (temperature=0)

---

## 1. Pipeline Stage Status

| Stage | Name | Status | Notes |
|-------|------|--------|-------|
| 1 | Ingest | ✅ Pass | CSV loads correctly, 6 tickets read |
| 2 | Prepare | ✅ Pass | Empty messages skipped, 6/6 passed |
| 3 | Prompt | ✅ Pass | System + user messages built for each ticket |
| 4 | Call Model | ✅ Pass | gpt-4.1-nano, temperature=0 |
| 5 | Parse Output | ⚠️ Fragile | Works now, but no fallback if model returns non-JSON |
| 6 | Deliver | ✅ Pass | Results collected into a list of dicts |

## 2. Errors Encountered (2/3 resolved)

| Stage | Error Type | Description | Resolved? |
|-------|------------|-------------|-----------|
| 1 — Ingest | `FileNotFoundError` | Typo in CSV path ('data/tickers.csv' instead of 'data/tickets.csv') | ✅ |
| 3 — Prompt | `KeyError` | Referenced ticket['subject'] instead of ticket['message'] | ✅ |
| 5 — Parse Output | `JSONDecodeError` | Model occasionally returns 'Sure! Here is...' prefix before JSON | ❌ |

## 3. Baseline Accuracy: 4/5 (80%)

| Ticket | Expected | Predicted | Match |
|--------|----------|-----------|-------|
| 101 | Billing | Billing | ✅ |
| 102 | Technical | Technical | ✅ |
| 103 | Account | General | ❌ |
| 104 | General | General | ✅ |
| 105 | Billing | Billing | ✅ |

## 4. Open Questions

1. The model confused 'Account' with 'General' on ticket 103. Would adding a few-shot example for Account changes fix this?
2. Parse Output has no fallback — what happens if the model wraps JSON in markdown fences (```json ... ```)?
3. We tested on 5 tickets. Is 5 enough to trust the accuracy, or do we need 20+?
4. Should we add a 'confidence' threshold and route low-confidence tickets to a human?
5. The current pipeline processes tickets one at a time. Will this be too slow for 1,000 tickets?


## Save the Checkpoint

A checkpoint is only useful if it's saved somewhere. Let's write it to a file.

In [7]:
import os

os.makedirs("data", exist_ok=True)
checkpoint_path = os.path.join("data", "checkpoint_v0.md")

# encoding="utf-8" is required on Windows: the default (cp1252) cannot
# write emoji such as ✅ / ❌ / ⚠️ that appear in checkpoint_text.
with open(checkpoint_path, "w", encoding="utf-8") as f:
    f.write(checkpoint_text)

print(f"✅ Checkpoint saved to {checkpoint_path}")

✅ Checkpoint saved to data\checkpoint_v0.md


## Why This Matters

Without a v0 checkpoint, your first pipeline run lives only in your head. A week later you can't remember:
- Which stages you already validated
- Which errors you fixed vs. which you worked around
- What your accuracy actually was
- What questions you meant to investigate next

The checkpoint captures all of this in five minutes. It's the difference between *"I think it was working"* and *"Here's exactly where we are."*

## 🎯 Summary

### Key Takeaways

1. **A v0 checkpoint has four sections** — pipeline stage status, errors encountered (mapped to stages), baseline accuracy, and open questions
2. **It combines all three module skills** — pipeline-stage awareness tells you *what works*, traceback reading tells you *what broke and where*, and baseline comparison tells you *how good it is*
3. **Open questions are the most valuable part** — they turn unknowns from forgotten worries into a written action list that guides your next iteration
4. **Save the artifact** — a checkpoint only helps if it's written down; a Markdown file you can share beats a mental note every time